# Randome Forest :

#### Pourquoi ce model est adapté ?

Le Random Forest est adapté car :

- **Données tabulaires** : Notre dataset est un tableau de stats numériques, c'est exactement le type de données où Random Forest est fort.

- **Déséquilibre des classes** : Les légendaires représentent ~12% du dataset. Le paramètre `class_weight='balanced'` permet de compenser ce déséquilibre sans avoir à modifier le dataset.

- **Relations non-linéaires** : Un légendaire n'est pas défini par une seule stat mais par une combinaison (ex: BST élevé ET Catch Rate faible ET pas de forme alternative). Random Forest capture naturellement ces combinaisons.

- **Feature Importance** : Random Forest nous donnera l'importance de chaque variable, ce qui permet de répondre à la question :
*"Quelles stats définissent vraiment un légendaire ?"*

- **Robuste aux outliers** : Certains Pokémon comme les méga-évolutions ont des stats extrêmes. Random Forest est peu sensible à ces valeurs extrêmes.

## 1. Import et cargements des donées

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

df = pd.read_csv("../../data/encoded/pokemon_ready.csv")
df.head()

## 2. Séparation des données

On separe les données du DataFrame en deux catégories :
- **X** : Toutes les colonnes (les statistiques)  qui servent à prédire
- **Y** : Ce qu'on veut prédire, c'est a dire estce que c'est un legendaire ?

Puis on les sépare en données d'entraînement et de test :
- **X_train / y_train** : Ce sont les données  d'entrainement
- **X_test / y_test** : Ce sont les données sur laquelle le modeles sera évalué

In [ ]:
# On sépare la colone cible des autres colonnes
# La colonne cible
y = df['Legendary']

# Toute les autres colonnes 
X = df.drop(columns=['Legendary'])

# Séparation en données d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

``random_state=42`` : les données seront toujours mélanger de la meme facon, pour qu'on obtienne les meme resultat a chque réexécution

``stratify=y`` : garantit les mêmes proportions de légendaires dans X_train/y_train 
et X_test/y_test — indispensable car les légendaires sont peu nombreux dans le dataset ! 

## 3. Entrainement du modéle

### 3.1 On cherche les meilleurs parametre pour entrainer le model

Pour ca on uttilise ``GridSearch``

On le parametre avec :
- cv=5 : il coupe les donnée train en 5 parties et teste chaque combinaison 5 fois
- scoring='f1' : on évalue sur le F1 et pas l'accuracy car on a peu de légendaires


In [ ]:
# Les paramètres à tester
param_grid = {
    'n_estimators': [100, 200, 300], # nombre d'arbre
    'max_depth': [None, 10, 20, 30], # profondeur maximale de chaque arbre
    'min_samples_split': [2, 5, 10] # nombre minimum d'échantillons pour diviser un nœud
}

grid_search = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    param_grid,
    cv=5,           # validation croisée en 5 parties
    scoring='f1',   # on optimise sur le F1 car jeux de données déséquilibrées
    n_jobs=-1       # parametre technique : utilise tous les coeurs du processeur (de l'ordinateur qui execute) pour aller plus vite
)

grid_search.fit(X_train, y_train)

print("Meilleurs paramètres :", grid_search.best_params_)

Les meilleurs parametre d'apres ``GridSearch`` sont...

### 3.2 On entraine le model

In [ ]:
# Entraînement du modèle avec les meilleurs paramètres
rf = RandomForestClassifier(
    **grid_search.best_params_, # injecte automatiquement les meilleurs paramètres trouvés par le GridSearch
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)

``**grid_search.best_params_`` : injecte automatiquement les meilleurs paramètres trouvés par le GridSearch
``class_weight='balanced'`` : compense le fait qu'on a peu de légendaires
``rf.fit()`` : c'est ici que le modèle apprend à partir de X_train et y_train

## 4. Evaluation

### 4.1 Matrice de confusion

In [ ]:
matrice_confusion = confusion_matrix(y_test, y_pred)
sns.heatmap(matrice_confusion, annot=True, fmt='d', cmap='RdPu',
            xticklabels=['Normal', 'Légendaire'],
            yticklabels=['Normal', 'Légendaire'])
plt.title("Matrice de confusion — Random Forest")
plt.ylabel("Réel")
plt.xlabel("Prédit")
plt.show()


Qu'est ce que ca veux dire  ?

### 4.2 Classification Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Normal', 'Légendaire']))

Qu'est ce que ca veux dire  ?

### 4.3 Les colonnes importantes

In [ ]:
# Colonnes Importantes
importances = pd.Series(rf_best.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True)

importances.plot(kind='barh', color='#e91e8c', figsize=(10, 8))
plt.title("Importance des variables — Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

Qu'est ce que ca veux dire  ?